# Downloading the data

Download
https://figshare.com/articles/dataset/processed_mutational_profiles/7312067

Unzip inside CROP/data/FORECasT

Unzip each file like "ST_APRIL_2017....zip" in its own folder
Should now have folders like:
CROP/data/FORECasT/ST_April_2017_K562_800x_6OA_DPI3_Old7/Oligos_51

Download Supplementary Data 1
https://www.nature.com/articles/nbt.4317#MOESM74 

Put inside CROP/data/FORECast


# Versions

We will need the following versions (shown as output for this cell):

In [1]:
import sys
import os
import pandas as pd
from collections import defaultdict
import re


print("Python version:", sys.version)
print("Pandas version:", pd.__version__)

Python version: 3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)]
Pandas version: 2.3.0


# Preprocess

We only care about the length difference

In [2]:
def parse_line_only_length(line):
    parts = line.strip().split('\t')
    if len(parts) < 2:
        return None

    label = parts[0]
    try:
        count = int(parts[1])
    except ValueError:
        return None

    # Find all operations: 'D' followed by digits OR 'I' followed by digits
    # This captures mixed cases like D14 ... I3 
    ops = re.findall(r'([DI])(\d+)', label)
    
    if not ops:
        return 0, 0

    net_length = 0
    for op, size in ops:
        size = int(size)
        if op == 'D':
            net_length -= size
        elif op == 'I':
            net_length += size

    return net_length, count

In [3]:
# all TargetSequences that are REVERSE, do reverse complement
def reverse_complement(seq):
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
    return ''.join(complement[base] for base in reversed(seq))



In [4]:
def get_target_sequence_from_oligo(df, oligo):
    row = df["ID"] == oligo
    if row.any():
        return df.loc[row, "TargetSequence"].values[0], int(df.loc[row, "PAM Index"].values[0])
    else:
        print(f"Oligo {oligo} not found in DataFrame.")
        return None, None

def parse_all_files_only_length(txt_files, df_seqs):
    all_oligos = defaultdict(lambda: defaultdict(int))

    for i, file_path in enumerate(txt_files):
        if i % 10 == 0:
            print(f"Parsing file {i + 1}/{len(txt_files)}: {file_path}")
        with open(file_path, "r") as f:
            current_oligo = None
            original_sequence = None

            for line in f:
                line = line.strip()
                if not line:
                    continue

                if line.startswith("@@@Oligo"):
                    current_oligo = line.replace("@@@", "").strip()
                    original_sequence, pam_index = get_target_sequence_from_oligo(df_seqs, current_oligo)
                elif current_oligo:
                    parts = line.split('\t')
                    if len(parts) < 3:
                        continue
                    count = int(parts[1])
                    sequence = parts[2]


                    repair_outcome, count = parse_line_only_length(line)
                    if repair_outcome:
                        all_oligos[current_oligo][repair_outcome] += count

    return all_oligos


In [5]:
def create_outcome_csv(all_oligos, df, min_diff, max_diff):
    outcome_range = list(range(min_diff, max_diff + 1))
    rows = []

    for i, (oligo, outcomes) in enumerate(all_oligos.items()):
        if i % 1000 == 0:
            print(f"Processed {i} oligos...")

        target_sequence = df.loc[df["ID"] == oligo, "TargetSequence"].values[0] if oligo in df["ID"].values else None
        pam_index = df.loc[df["ID"] == oligo, "PAM Index"].values[0] if oligo in df["ID"].values else None
        row = {"Oligo": oligo, "TargetSequence": target_sequence, "PAM Index": pam_index}
        for outcome in outcome_range:
            row[str(outcome)] = outcomes.get(outcome, 0)
        rows.append(row)

    print(f"Finished processing {len(all_oligos)} oligos.")
    outcome_df = pd.DataFrame(rows)
    return outcome_df

In [6]:
def natural_sort_key(s):
    """Split string into parts of digits and non-digits for natural sorting."""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def sort_oligo_df_naturally(df):
    """Sort the DataFrame by 'Oligo' column in natural (human-readable) order."""
    return df.sort_values(by="Oligo", key=lambda col: col.map(natural_sort_key)).reset_index(drop=True)

In [7]:
def get_all_subfolders(folder):
    """Get all subfolders in the given folder, returning full paths."""
    return [os.path.join(folder, f) for f in os.listdir(folder) if os.path.isdir(os.path.join(folder, f))]


def get_all_txt_files(subfolders):
    """Get all .txt files in the given subfolders, returning full paths."""
    txt_files = []
    for subfolder in subfolders:
        for file in os.listdir(subfolder):
            if file.endswith('.txt'):
                txt_files.append(os.path.join(subfolder, file))
    return txt_files

In [8]:
def create_forecast_length_only_dataset(folders, save_path, df_seqs):
    subfolders = [get_all_subfolders(folder) for folder in folders]
    txt_files = [get_all_txt_files(subfolder) for subfolder in subfolders]
    # flatten the list of lists
    txt_files = [item for sublist in txt_files for item in sublist]
    print(f"Found {len(txt_files)} .txt files in {len(subfolders)} subfolders.")

    all_oligos = parse_all_files_only_length(txt_files, df_seqs)


    # get min and max length diffs
    min_diff = float('inf')
    max_diff = float('-inf')
    for oligo, diffs in all_oligos.items():
        for diff in diffs.keys():
            if diff < min_diff:
                min_diff = diff
            if diff > max_diff:
                max_diff = diff
    print(f"Minimum length difference: {min_diff}")
    print(f"Maximum length difference: {max_diff}")


    outcome_df = create_outcome_csv(all_oligos, df_seqs, min_diff, max_diff)
    df_sorted = sort_oligo_df_naturally(outcome_df)
    # Rename TargetSequence to sequence and PAM Index to PAM position
    df_sorted.rename(columns={"TargetSequence": "sequence", "PAM Index": "PAM position"}, inplace=True)

    # remove the 0 column if it exists
    if '0' in df_sorted.columns:
        df_sorted.drop(columns=['0'], inplace=True)

    # remove the Oligo column
    df_sorted.drop(columns=['Oligo'], inplace=True)

    df_sorted.to_csv(save_path, index=False)
    print(f"Outcome DataFrame saved to {save_path}")

In [9]:
SEQS_PATH = "../data/FORECasT/41587_2019_BFnbt4317_MOESM72_ESM.txt"

# open as .tab file
df_seqs = pd.read_csv(SEQS_PATH, sep="\t", header=0)
# Keep the following columns:
columns_to_keep = [
    "ID",
    "Guide Sequence",
    "TargetSequence",
    "PAM Index",
    "Strand"
]
df_seqs = df_seqs[columns_to_keep]

df_seqs.head()

,ID,Guide Sequence,TargetSequence,PAM Index,Strand
0,Oligo1,GTGCGATCCGGAGTAGTTCT,CAATCCGTCTGTCTGTTCGAAGTAGTGTGTTACCTTTTGCGATCCG...,56,FORWARD
1,Oligo2,GGTTGAAAGTCTATAGTGGT,AAATGCGTAACAAAAACAAGCTCGGTATCTGGCCTATTCCACGCCA...,49,REVERSE
2,Oligo8,GACAATGCTCGGCAAATACC,TAACGACGTCAACCCATTCGCATACAATGCTCGGCAAATACCTGGT...,42,FORWARD
3,Oligo11,GGTCGGTAGCCAAGAAGGAA,TAAGATCAAGTTGATTGGTTCCGGTCGGTAGCCAAGAAGGAAGGGT...,42,FORWARD
4,Oligo12,GCGGAGGTCATTACTCATTC,GAAAGGCCGCAAGTTGCCCCATATAAACTGGGTCTCTCGGAGGTCA...,56,FORWARD


In [10]:
# treat reverse complement sequences

for idx, row in df_seqs.iterrows():
    if row["Strand"] == "REVERSE":
        length = len(row["TargetSequence"])
        new_seq = reverse_complement(row["TargetSequence"])
        new_pam_index = length - row["PAM Index"] 
        df_seqs.at[idx, "TargetSequence"] = new_seq
        df_seqs.at[idx, "PAM Index"] = new_pam_index

# Procsss per cell

In [11]:
LIB_A_FOLDER = "../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_June_2017_K562_800x_LV7B_DPI7"
SAVE_PATH = "../data/FORECasT_K562.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_June_2017_K562_800x_LV7A_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 

In [12]:
LIB_A_FOLDER = "../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_June_2017_HAP1_LV7B_DPI7"
SAVE_PATH = "../data/FORECasT_HAP1.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_June_2017_HAP1_LV7A_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data/FORECasT/ST_June_2017_H

In [13]:
LIB_A_FOLDER = "../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_June_2017_E14TG2A_LV7B_DPI7"
SAVE_PATH = "../data/FORECasT_mESC.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_June_2017_E14TG2A_LV7A_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data

In [14]:
LIB_A_FOLDER = "../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_June_2017_CHO_LV7B_DPI7"
SAVE_PATH = "../data/FORECasT_CHO.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data/FORECasT/ST_June_2017_CHO_LV7A_

In [15]:
LIB_A_FOLDER = "../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_June_2017_BOB_LV7B_DPI7"
SAVE_PATH = "../data/FORECasT_BOB.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data/FORECasT/ST_June_2017_BOB_LV7A_

In [16]:
LIB_A_FOLDER = "../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_Feb_2018_TREX2_12NB_DPI7"
SAVE_PATH = "../data/FORECasT_K562_TREX2.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_Feb_2018_TREX2_12NA_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data/FORECasT/ST_Feb_2018_TR

In [17]:
LIB_A_FOLDER = "../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_Feb_2018_2A_TREX2_12NB_DPI7"
SAVE_PATH = "../data/FORECasT_K562_2A_TREX2.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_Feb_2018_2A_TREX2_12NA_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data

In [18]:
LIB_A_FOLDER = "../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7"
LIB_B_FOLDER = "../data/FORECasT/ST_Feb_2018_eCAS9_12NB_DPI7"
SAVE_PATH = "../data/FORECasT_K562_eCAS9.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER, LIB_B_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 3296 .txt files in 2 subfolders.
Parsing file 1/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/3296: ../data/FORECasT/ST_Feb_2018_eCAS9_12NA_DPI7\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsing file 81/3296: ../data/FORECasT/ST_Feb_2018_eC

RPE1 is noisier (only 1 replicate)

In [19]:
LIB_A_FOLDER = "../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec"
SAVE_PATH = "../data/FORECasT_RPE1.csv"
create_forecast_length_only_dataset(folders= [LIB_A_FOLDER], save_path=SAVE_PATH, df_seqs=df_seqs)

Found 1648 .txt files in 1 subfolders.
Parsing file 1/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_0\Oligos_0-49_processedindels.txt
Parsing file 11/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_0\Oligos_500-549_processedindels.txt
Parsing file 21/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_1\Oligos_1000-1049_processedindels.txt
Parsing file 31/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_1\Oligos_1500-1549_processedindels.txt
Parsing file 41/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_10\Oligos_10000-10049_processedindels.txt
Parsing file 51/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_10\Oligos_10500-10549_processedindels.txt
Parsing file 61/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_11\Oligos_11000-11049_processedindels.txt
Parsing file 71/1648: ../data/FORECasT/ST_Feb_2018_RPE1_500x_7B_DPI7_dec\Oligos_11\Oligos_11500-11549_processedindels.txt
Parsi